# 02 — Calidad y preparación de datos

**Responsable:** Ignacio Silva  
**Etapa:** Data Preparation (CRISP-DM)  
**Objetivo:** construir una versión reproducible de una fila por canción, sin modificar la fuente en `data/raw/`.

## Punto de partida y controles

La etapa anterior identificó 114.000 filas, tres celdas nulas en una misma observación, una duración no positiva y repetición de `track_id`. Aquí se verifican esos hallazgos antes de transformar. Las reglas reutilizables viven en `src/spotify_popularity/`; el notebook solo deja evidencia y narrativa.

In [1]:
from spotify_popularity.analysis.quality import dataset_overview
from spotify_popularity.config import PROJECT_PATHS
from spotify_popularity.data.loader import load_raw_dataset
from spotify_popularity.data.preparation import prepare_modeling_dataset

In [2]:
raw_data = load_raw_dataset()
overview = dataset_overview(raw_data)
overview

{'row_count': 114000,
 'column_count': 21,
 'missing_cells': 3,
 'exact_duplicate_rows': 0,
 'unique_track_ids': 89741,
 'repeated_track_ids': 16641,
 'rows_with_repeated_track_id': 40900,
 'extra_track_id_rows': 24259}

## Decisiones de limpieza

1. Se elimina `Unnamed: 0` solo del resultado derivado, pues es un índice de exportación.
2. Se excluye la única fila sin artista, álbum y nombre de pista, cuya duración es cero. No se imputan textos sin evidencia.
3. Se consolida cada `track_id` en una fila. Los atributos musicales, de álbum y textuales fueron invariantes para las repeticiones; la preparación se detiene si esa condición deja de cumplirse.
4. Se conservan todos los géneros como `track_genres`, con etiquetas únicas, ordenadas y separadas por `|`.
5. En los IDs con valores de popularidad distintos se usa la mediana, para no privilegiar una asignación de género por orden de archivo. La popularidad procesada puede ser decimal.

In [3]:
clean_data, summary = prepare_modeling_dataset(raw_data)
summary

PreparationSummary(input_rows=114000, invalid_rows_removed=1, output_rows=89740, repeated_track_ids_consolidated=24259)

## Validación del resultado

El resultado esperado conserva 89.740 canciones únicas. Se validan identificadores únicos, ausencia de nulos, duración positiva y géneros presentes antes de persistir el archivo regenerable.

In [4]:
assert clean_data.shape == (89_740, 20)
assert clean_data["track_id"].is_unique
assert clean_data.isna().sum().sum() == 0
assert clean_data["duration_ms"].gt(0).all()
assert clean_data["track_genres"].str.len().gt(0).all()
clean_data.head()

,track_id,artists,album_name,track_name,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,popularity,track_genres
0,0000vdREvCVMxbQTkS888c,Rill,Lolly,Lolly,160725,True,0.910,0.374,8,-9.844,0,0.1990,0.075700,0.00301,0.1540,0.432,104.042,4,44.0,german
1,000CC8EParg64OmTxVnZ0p,Glee Cast,Glee Love Songs,It's All Coming Back To Me Now (Glee Cast Vers...,322933,False,0.269,0.516,0,-7.361,1,0.0366,0.406000,0.00000,0.1170,0.341,178.174,4,47.0,club
2,000Iz0K615UepwSJ5z2RE5,Paul Kalkbrenner;Pig&Dan,X,Böxig Leise - Pig & Dan Remix,515360,False,0.686,0.560,5,-13.264,0,0.0462,0.001140,0.18100,0.1110,0.108,119.997,4,22.0,minimal-techno
3,000RDCYioLteXcutOjeweY,Jordan Sandhu,Teeje Week,Teeje Week,190203,False,0.679,0.770,0,-3.537,1,0.1900,0.058300,0.00000,0.0825,0.839,161.721,4,62.0,hip-hop
4,000qpdoc97IMTBvF8gwcpy,Paul Kalkbrenner,Zeit,Tief,331240,False,0.519,0.431,6,-13.606,0,0.0291,0.000964,0.72000,0.0916,0.234,129.971,4,19.0,minimal-techno


In [5]:
PROJECT_PATHS.create_generated_directories()
output_path = PROJECT_PATHS.data_processed / "spotify_tracks_clean.csv"
clean_data.to_csv(output_path, index=False)
output_path

WindowsPath('C:/Users/yvl/Documents/GitHub/spotify-popularity-analysis/data/processed/spotify_tracks_clean.csv')

## Entrega y limitaciones

El archivo `data/processed/spotify_tracks_clean.csv` queda preparado para EDA y modelamiento sin modificar el CSV original. `track_genres` es multietiqueta: si la siguiente etapa expande una canción por género, no debe interpretar esas filas como canciones distintas. La popularidad sigue siendo una fotografía histórica, no una medida actual de Spotify. El detalle de decisiones y métricas queda en `docs/progress/ignacio_silva.md`.